In [1]:
#!pip install pandas faiss-cpu sentence-transformers


In [2]:
# data  
# proper paragraphs 
# clear word word 
# numerical (vectors) 
# vector_database 
# llm (apikey)  (llm) 
# search (text) -- (text) numerical 
# vectordata (train)  proper answer 

In [3]:
import pandas as pd   

def load_and_chunk_csv(csv_path):
    df = pd.read_csv(csv_path)
    chunks = []

    for idx, row in df.iterrows():
        row_text = ""
        for col in df.columns:
            row_text += f"{col}: {row[col]}\n"
        chunks.append(row_text.strip())

    return chunks  


In [4]:
from sentence_transformers import SentenceTransformer

def generate_embeddings(text_chunks):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(text_chunks) 
    return embeddings  


c:\Users\Admin\anaconda3\envs\pooji\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import faiss  
import numpy as np 

def store_in_faiss(embeddings, save_path="crop_vectors.faiss"):
    dim = embeddings[0].shape[0]
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings))
    faiss.write_index(index, save_path)
    print(f"FAISS index saved to {save_path}")
    return index


In [6]:
def inspect_faiss_index(index, num_vectors=5):
    print("FAISS index info:")
    print("Total vectors:", index.ntotal)
    
    # View the first few vectors (optional)
    _, I = index.search(index.reconstruct_n(0, num_vectors), k=1)
    print(f"Top {num_vectors} vectors indices (self-matched):", I.flatten()) 


In [7]:
if __name__ == "__main__":
    csv_path = "labeled_data.csv"  # replace with your actual file

    # Step 1: Load + chunk
    chunks = load_and_chunk_csv(csv_path)
    print("Sample chunk:\n", chunks[0], "\n")

    # Step 2: Embeddings
    embeddings = generate_embeddings(chunks)

    # Step 3: Store in FAISS
    index = store_in_faiss(embeddings, "csv_vectors.faiss")

    # Step 4: Inspect
    inspect_faiss_index(index)  


Sample chunk:
 Unnamed: 0: 0
count: 3
hate_speech: 0
offensive_language: 0
neither: 3
class: 2
tweet: !!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out... 

FAISS index saved to csv_vectors.faiss
FAISS index info:
Total vectors: 24783
Top 5 vectors indices (self-matched): [0 1 2 3 4]
